In [1]:
import numpy as np
import pandas as pd

In [2]:
FOLD = 7

In [ ]:
# flat_regions_path = f"/scratch1/smaruj/genomic_flat_regions/flat_regions_chrom_states_tsv/fold{FOLD}_selected_genomic_windows_centered_chrom_states.tsv"

In [3]:
flat_regions_path = f"/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/suppressing_CTCFs/fold{FOLD}_with_positions.tsv"

In [4]:
df = pd.read_csv(flat_regions_path, sep="\t")

In [5]:
import torch

In [6]:
from typing import Optional

In [7]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
# def splice_and_to_string(main_path: str,
#                          slice_path: str,
#                          replace_bin: int = 320,
#                          bin_size: int = 2048,
#                          trim_bp: int = 131072,
#                          alphabet: str = "ACGT",
#                          device: Optional[str] = None):
#     """
#     Load a one-hot DNA tensor (shape: [4, L]), replace one 2048-bp bin in the middle,
#     trim `trim_bp` bases from both ends, and convert to an A/C/G/T string.

#     Parameters
#     ----------
#     main_path : str
#         Path to the main .pt tensor (e.g. 'chr11_65677312_66988032_X.pt').
#     slice_path : str
#         Path to the .pt tensor providing the replacement slice
#         (e.g. 'chr11_65677312_66988032_slice.pt').
#     replace_bin : int
#         Index of the 2048-bp bin to replace (default: 320).
#     bin_size : int
#         Size of each bin in bp (default: 2048).
#     trim_bp : int
#         Number of bp to cut from each side after replacement (default: 131072).
#     alphabet : str
#         Order of nucleotides corresponding to the one-hot channels (default: "ACGTN").
#         Change if your tensor channel order differs.

#     Returns
#     -------
#     seq_str : str
#         The trimmed sequence converted to letters.
#     """

#     main = torch.load(main_path, map_location=device)
#     slc  = torch.load(slice_path, map_location=device)

#     if main.dim() != 3 or main.size(1) != len(alphabet):
#         raise ValueError(f"Expected main tensor of shape [4, L]. Got {tuple(main.shape)}")

#     start = replace_bin * bin_size
#     end   = (replace_bin + 1) * bin_size

#     if end > main.size(2):
#         raise ValueError(f"Replacement range [{start}:{end}] exceeds sequence length {main.size(2)}")
    
#     # Check slice sizes
#     if slc.dim() != 3 or slc.size(1) != len(alphabet) or slc.size(2) != (end - start):
#         raise ValueError(
#             f"Slice tensor must have shape [1, {len(alphabet)}, {end-start}], got {tuple(slc.shape)}"
#         )

#     # Trim
#     L = main.size(2)
#     if 2 * trim_bp >= L:
#         raise ValueError(f"trim_bp ({trim_bp}) is too large for sequence length {L}")
#     trimmed = main[:, :, trim_bp : L - trim_bp]

#     # Convert to string
#     # Argmax over channels -> indices 0..3
#     idx = trimmed.argmax(dim=1).cpu().numpy()
    
#     letters = np.array(list(alphabet))
#     seq_arr = letters[idx]

#     if seq_arr.ndim > 1:
#         seq_arr = seq_arr.ravel()
    
#     seq_str = "".join(seq_arr.tolist())

#     return seq_str


In [8]:
import ast
import random
from typing import Optional, List, Tuple

In [9]:
def insert_sine_b2_with_constraints(
    seq_str: str, 
    ctcf_locs_raw: str, 
    bin_size: int = 2048, 
    center_bin_idx: int = 256
) -> str:
    """
    Inserts 3 copies of SINE B2 into the center bin of seq_str, 
    avoiding CTCF sites +/- 15bp.
    """
    sine_b2 = ("GGGGCTGGAGAGATGGCTCAGCGGTTAAGAGCACTGACTGCTCTTCCAGAGGTCCTGAGTTCAATTCCC"
               "AGCAACCACATGGTGGCTCACAACCATCTGTAATGGGATCTGATGCCCTCTTCTGGTGTGTCTGAAGACA"
               "GCTACAGTGTACTCACATACATAAATAAATAAATAAATCTTTAAAAAAAAAAAAAA")
    sine_len = len(sine_b2)
    
    # 1. Parse CTCF locations
    try:
        # Convert string representation of list of tuples to actual list
        ctcf_locs = ast.literal_eval(ctcf_locs_raw) if isinstance(ctcf_locs_raw, str) else ctcf_locs_raw
    except:
        ctcf_locs = []

    # 2. Define the absolute bounds of the center bin within seq_str
    # Based on your description: center bin is the 256th bin (0-indexed)
    bin_start_in_seq = center_bin_idx * bin_size
    bin_end_in_seq = bin_start_in_seq + bin_size

    # 3. Create forbidden ranges (Relative to the start of the full seq_str)
    forbidden = []
    for (rel_start, rel_end) in ctcf_locs:
        # Map relative bin coords to absolute sequence coords
        abs_start = bin_start_in_seq + rel_start - 15
        abs_end = bin_start_in_seq + rel_end + 15
        forbidden.append((abs_start, abs_end))

    # 4. Insertion Logic
    seq_list = list(seq_str)
    placed_count = 0
    attempts = 0
    max_attempts = 500
    
    # Keep track of where we've already put a SINE B2 to avoid overwriting ourselves
    placed_ranges = []

    while placed_count < 3 and attempts < max_attempts:
        attempts += 1
        # Pick a random start position within the 2048bp bin
        # Ensure the whole SINE B2 fits inside the bin
        proposal_start = random.randint(bin_start_in_seq, bin_end_in_seq - sine_len)
        proposal_end = proposal_start + sine_len
        
        # Check collision with CTCF zones or previously placed SINEs
        collision = False
        for (f_start, f_end) in (forbidden + placed_ranges):
            if not (proposal_end <= f_start or proposal_start >= f_end):
                collision = True
                break
        
        if not collision:
            # Insert (overwrite) the sequence at this location
            seq_list[proposal_start:proposal_end] = list(sine_b2)
            placed_ranges.append((proposal_start, proposal_end))
            placed_count += 1

    if placed_count < 3:
        print(f"Warning: Only placed {placed_count} SINE B2 sequences due to space constraints.")

    return "".join(seq_list)

def splice_and_to_string(main_path: str,
                         ctcf_locs_raw: str, # Added this param
                         bin_size: int = 2048,
                         trim_bp: int = 131072,
                         alphabet: str = "ACGT",
                         device: Optional[str] = None):
        
    main = torch.load(main_path, map_location=device)
    
    if main.dim() != 3 or main.size(1) != len(alphabet):
        raise ValueError(f"Expected main tensor of shape [4, L]. Got {tuple(main.shape)}")
    
    L = main.size(2)
    trimmed = main[:, :, trim_bp : L - trim_bp]
    idx = trimmed.argmax(dim=1).cpu().numpy()
    letters = np.array(list(alphabet))
    seq_arr = letters[idx]
    if seq_arr.ndim > 1: seq_arr = seq_arr.ravel()
    
    raw_seq_str = "".join(seq_arr.tolist())

    # NEW: Apply the SINE B2 insertion
    final_seq = insert_sine_b2_with_constraints(
        raw_seq_str, 
        ctcf_locs_raw, 
        bin_size=bin_size, 
        center_bin_idx=256 # Adjust if 256 is not the relative center index after trimming
    )

    return final_seq

In [10]:
def save_to_fasta(seq: str, fasta_path: str, header: str = "sequence"):
    """
    Save a sequence string to a FASTA file.
    
    Parameters
    ----------
    seq : str
        The sequence (A/C/G/T/N).
    fasta_path : str
        Path to save the FASTA file.
    header : str
        The FASTA header (default: "sequence").
    """
    with open(fasta_path, "w") as f:
        f.write(f">{header}\n")
        # Write 80 characters per line (FASTA convention)
        for i in range(0, len(seq), 80):
            f.write(seq[i:i+80] + "\n")

In [ ]:
df.columns

In [11]:
for i, row in enumerate(df.itertuples(index=False)):
    
    print(f"seq {i}")
    
    chrom, start, end = row.chrom, row.centered_start, row.centered_end
    ctcf_loc = row.ctcf_motif_locs
    
    seq = splice_and_to_string(main_path = f"/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/suppressing_CTCFs/ohe_X/fold{FOLD}/{chrom}_{start}_{end}_X.pt",
                               ctcf_locs_raw = ctcf_loc,
                         device = device)
    
    save_to_fasta(seq = seq, fasta_path=f"/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/alpha_genome_validation/ctct_suppressing/fold{FOLD}_original_3sineb2/{chrom}_{start}_{end}.fasta", header=f"{chrom}_{start}_{end}")

seq 0
seq 1
seq 2
seq 3
seq 4
seq 5
seq 6
seq 7
seq 8
seq 9
seq 10
seq 11
seq 12
seq 13
seq 14
seq 15
seq 16
seq 17
seq 18
seq 19
seq 20
seq 21
seq 22
seq 23
seq 24
seq 25
seq 26
seq 27
seq 28
seq 29
seq 30
seq 31
seq 32
seq 33
seq 34
seq 35
seq 36
seq 37
seq 38
seq 39
seq 40
